# Ajustar y generalizar

**Explorador de Hespérides · Capítulo 3**

Ampliación programada sobre los conceptos de los notebooks D2L de este capítulo.

Compara la curva verdadera, las observaciones y el modelo. Grado y regularización actúan de formas distintas. Aquí la validación orienta la selección; no es un test final independiente. Los puntos y errores son resultados calculados, con semilla fija.

![Ilustración conceptual](../recursos/ilustraciones/capitulo_3.png)

*Ilustración conceptual generada con ImageGen. Los resultados cuantitativos son los del código.*

In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import ipywidgets as widgets
from ipywidgets import interact

torch.manual_seed(42)
np.random.seed(42)
torch.set_num_threads(2)
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False, "animation.embed_limit": 40})


In [ ]:

# La función verdadera se conoce porque aquí los datos son sintéticos.
rng = np.random.default_rng(42)
x_train = np.sort(rng.uniform(-1, 1, 24)); x_val = np.sort(rng.uniform(-1, 1, 80))
y_train = np.sin(np.pi*x_train) + rng.normal(0,.22,len(x_train))
y_val = np.sin(np.pi*x_val) + rng.normal(0,.22,len(x_val))
xx = np.linspace(-1,1,300)

def ajuste(grado, regularizacion):
    # Base de Legendre para no confundir sobreajuste con una Vandermonde mal condicionada.
    A = np.polynomial.legendre.legvander(x_train, grado)
    penalizacion = np.eye(grado+1);penalizacion[0,0] = 0
    sistema = np.vstack([A, np.sqrt(regularizacion*len(A))*penalizacion])
    objetivo = np.concatenate([y_train, np.zeros(grado+1)])
    return np.linalg.lstsq(sistema, objetivo, rcond=None)[0]

def ver_generalizacion(grado=5, regularizacion=.001):
    w = ajuste(grado, regularizacion)
    predecir = lambda x: np.polynomial.legendre.legvander(x, grado) @ w
    fig, axes = plt.subplots(1,2,figsize=(11,4))
    axes[0].scatter(x_train,y_train,color='#087E8B',label='Entrenamiento')
    axes[0].scatter(x_val,y_val,facecolors='none',edgecolors='#D99B18',alpha=.5,label='Validación')
    axes[0].plot(xx,np.sin(np.pi*xx),'k--',label='Señal verdadera')
    axes[0].plot(xx,predecir(xx),color='#775DA6',label='Modelo')
    axes[0].set(ylim=(-2,2),title=f'Grado {grado} · λ={regularizacion:g}',xlabel='x',ylabel='y')
    errores=[]
    for d in range(1,19):
        wd=ajuste(d,regularizacion)
        errores.append([np.mean((np.polynomial.legendre.legvander(x,d)@wd-y)**2)
                        for x,y in [(x_train,y_train),(x_val,y_val)]])
    axes[1].semilogy(range(1,19),errores)
    axes[1].axvline(grado,color='black',alpha=.3)
    axes[1].set(xlabel='Grado',ylabel='MSE',title='Ajuste y generalización')
    axes[1].legend(['Entrenamiento','Validación']);axes[0].legend(fontsize=8)
    fig.tight_layout();plt.show()

interact(ver_generalizacion,grado=widgets.IntSlider(value=5,min=1,max=18,description='Grado',continuous_update=False),
         regularizacion=widgets.FloatLogSlider(value=.001,base=10,min=-7,max=1,step=.5,
                                              description='λ',continuous_update=False));


## Vista de referencia

Esta figura conserva el estado inicial también en una exportación sin kernel. Los controles anteriores se utilizan en Jupyter.

In [ ]:
ver_generalizacion(5,.001)

## Comprobación

Modifica un control cada vez y describe qué cambia y qué permanece constante. Compara tu observación con las preguntas del capítulo.